# MolGPT reference benchmark

The model is fine-tuned on 25% and 100% of `curation v2`. Chemical metrics are calculated after training with the frozen evaluator.


In [ ]:
%pip install -q "transformers==4.46.3" "tokenizers>=0.20,<0.21" accelerate rdkit

In [ ]:
import base64
import gzip
import json
import math
import random
import shutil
from pathlib import Path
from zipfile import ZipFile

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm


TRAIN_B64 = """H4sIAAAAAAAC/8VbuY7bMBDt/SUkAhUkndKNB3Bpqw9SqV1s/r8L75sUObSTBWzTksVjzveGXDjYIf8I3MnB5ccXzd+4fmve+mYXAAKUyA7YQR40eTH1ql0FYgY+GH0yUH/soq8QPRH1x6lrPMtLVD4pB2ZmbKBPAu4WdiXuedmvmons9CCbuSg/hb5g3oR84zuXk1azFdUehX5r3gJOdV/xYHbibhQOZkriuB5ETWInx089xZ80NPiVAgjf2dfdC1YNZZs7AStk6hrRuDuQm5MjUDOLTGP8QRtX7bjAZOMVdLdXVPbSP4IL6F/LFqh39ZI/kZORhmLE2hUqSLn4B31PstOnbmlLnjBEJ6JIHnKEm9Ns9oB40PpVUNrJbST7fgOhZS94KjZQlsSsR8h+vCp8Q7kJcLOwilisqhu33Cgv7ysM3KqnZHVxHexOyNNz8Usyfe3eWO/s5ezkxfSvILZQa01C3lV2KEe/FhO8qlf1quoVhJFk7mFGEY3QIw7rd1d9+UpDgwsTgr7Sxd6JcqbQT80PkOEp1yc09YkwkVTUiYi0eQJHxgU1UxVyONEOoaWqgrVt4uNnEKwKZjL0vCCKd3qtbhEMaPY9NT8T4+/CaXdzDaljxuvWmMdpv37simqSI25JRKlgIUU/YoduadIoGvjTKDxye2nRcCpT+jnjQ1pftmqQa3BSHO4pdtraQE5EuUsGjxwKzjYFPH2a5uvStFlsYrU7ysdPHEN/35zDU4/QZocpIm3H5JU8TUa2g+YN3rY/AyK5T5fw4q20cWK5QtuDoIDHpHkYBY0OdMxXn1gjmYgHNhwk8YDBr/v2u/FY4/Kv/cdvFlB/wF1poHstRDmbxblRso7h0gSudOcJbbi7sY0QreOyAEvsbAIQ+yZr9OgGk6gURjIYECwK29FIcpLm3RyqNRh2aFUaQGFXdr7ocVAgISk7JnmAYV7cw3e4PEDeIg9nUbqtva4KDaNfSMt5koBpdCIxOiiQC3HMgvpGBFxkMuKYdXDnC26UqQ54ln9VxOCNLuSafIRWcYJXRXOeaqfITC2fJIlKaD5amYqV8BhWt7R0waBNnGxWR3biQJJHSzQntAXiHlajfcqadD3vuYzpftxJmGBqNWZZ9smDbdbRZlCBe5zxOlHcAlGsiPaq39q3VPebOMn7gZ6j84PlKkVOZH14A7u3egy2j5J5gZEVSq5z1jb8nXB0OPdvI1aDF0ItBALsqHr5nPP3MauxoD/IUoOIrR2T9zNryFVkJvh9ykmC/5NJ1hNZSebvoRRr46X1CSc4L0kO3RigdCrT/O5Xiy9uoet+Se7PhcxvcVJyafUGfN47MFW+8ZLcSW0PBfJslW+2pO9LeyxHDYhyiKn0VTlcXqW/N324W74rc5mDntj9g7Otha5LnJfJhovRWXFsqnI1jAwG61WjMUczJV9oSkMgfQO065UKlOAbcLxSI8BX4vBPphsGvTpTK9ltEQd+M1wZRiCnJRmc/kKd5USJ/TLKB80rxl2lSVlIlG0X6ZlZMIWsvBdllRlnDOG1XlCpB7KmfRr5IyshZc0Iy6emKxKhWlSw6Zitx21j0+2bDW6N7S7JhS1bClBmwaT6zPu9lX9MzSJl79zydDf7hIljcWKGOtoMvJPpfYFJZfg1EcYHKQxHvziAWUTCjKGXZw8Ccd5E2MWvF/Rb3/sEedji6rQ4ScKIrZzqVGeiecoNq8jSM8Ao6aCV25bC0wkWFiI7EJ+XrK5W8UmLKGLxdH/PRbE0y8ziCHeDtw+acaKVgw8ztCXcvQItvmck5oj5y4xXNDYCayeuopqi8CjK8RZ3XEusn8WqbuCer6CKL3M6Xh7ZgBVQP3Suo5LyNl8/OT86tMKjp4Q5gKbz6tIQel2LKJ1NvdJR13eo8oCbHy6BaHsn2V1BqKe1z4LRdHaIzRRL2w/07vkq7US2P4FxGOFMADrkXmEV2lU2TBCzr26dNE+aFbireuBVx6qyPjLLYCfL38gNlHqNzTHut+bnMqYnVfKJsNzCROhTMZXTywO183mws4qDs8OdLcLJaPdminOOj5VUB6jJwtmkNwDgGSK+WA8M5+SXgRf09hXH4tTgppSmcJ8IBel/NEzwomIrNKzuGdP63k5PKHCh2F6xKfQZZH/uN327Gh0ssqazPfqpHdbOzvxJhlw6T4n/L5liQzU/ST+/yRZN679F2DzmvQs3WYRdO3lSPbUzGpcmYEAWnlaTa01bH3fkAfVMnAPqKOUfpPoSDf8F6H8v7bs2AAA="""
VALIDATION_B64 = """H4sIAAAAAAAC/51UsdLDIAje8yR6/3UA5yzhzrHJ3uvk2vefC1gTYyTNX9pwCskHfKCUICVHnh+kx3R7uhSSi373hOgN82P5e6JLqBBZIat2LwsYaI21HD7h6OQmjUIv36qg6swlEnCNRCM54ihQRQFx7feTvtGUBdF3rQPNk5sVRAQUXlerqSqRCxp/r4iEolAlG8i3+5ECiVRl392WXq6tgw+qLNdAr71zyr06voysFqlTBP26qKBYXGbhOs2Gtcs2wWdm7QzPfDK90MGNdQq5/iYhjJ96UUdZBP22KANTVf/vRpy5vk62UGMmTgtUFPIXhVtMMAvQz+dQIAkZkX+iO8MkpHU5s9tUDxQILGnW/L8D7cf98lE+5oUjo5abxJezNRodvpJtM0Y6DCuChXrsl2EtlRzP/GUSTu7jbZ+ZVrM1UVaGeil1WNiTcB3wjIg3Yd+jPskGAAA="""


def unpack(value):
    return gzip.decompress(base64.b64decode(value)).decode("utf-8").splitlines()


train_order = unpack(TRAIN_B64)
validation_smiles = unpack(VALIDATION_B64)

FRACTIONS = {"25": 42, "100": 166}
TRAINING_SEEDS = [11, 22, 33]
GENERATION_SEEDS = [101, 202, 303]
N_SAMPLES = 1000
TARGET_EXPOSURES = 8000
MAX_LENGTH = 202

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
print("Train:", len(train_order), "| validation:", len(validation_smiles))

## Model and training


In [ ]:
from rdkit import Chem, rdBase
from torch.utils.data import DataLoader, Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer


MODEL_ID = "jonghyunlee/MolGPT_long_context_pretrained-by-ZINC15"
MODEL_REVISION = "2ec4688c6014948c9f65fcec49f056decb59d967"
RESULT_DIR = Path("results/external_molgpt")
WEIGHT_DIR = Path("models/external_molgpt")
RESULT_DIR.mkdir(parents=True, exist_ok=True)
WEIGHT_DIR.mkdir(parents=True, exist_ok=True)

BATCH_SIZE = 4
LEARNING_RATE = 5e-5
EVAL_EVERY = 100
MAX_STEPS = TARGET_EXPOSURES // BATCH_SIZE
TEMPERATURE = 0.9
TOP_P = 0.90
RANDOM_SMILES_PER_MOLECULE = 5


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def load_model():
    tokenizer = AutoTokenizer.from_pretrained(
        MODEL_ID,
        revision=MODEL_REVISION,
        use_fast=True,
    )
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        revision=MODEL_REVISION,
        torch_dtype=torch.float32,
    )
    model.resize_token_embeddings(len(tokenizer))
    model.config.bos_token_id = tokenizer.bos_token_id
    model.config.eos_token_id = tokenizer.eos_token_id
    model.config.pad_token_id = tokenizer.pad_token_id
    model.generation_config.bos_token_id = tokenizer.bos_token_id
    model.generation_config.eos_token_id = tokenizer.eos_token_id
    model.generation_config.pad_token_id = tokenizer.pad_token_id
    return tokenizer, model


def roundtrip_ok(smiles, tokenizer):
    ids = tokenizer(smiles, add_special_tokens=False)["input_ids"]
    decoded = "".join(tokenizer.decode(ids, skip_special_tokens=True).split())
    return decoded == smiles


def randomized_smiles(smiles_values, seed):
    rdBase.SeedRandomNumberGenerator(seed)
    rows = []
    for smiles in smiles_values:
        mol = Chem.MolFromSmiles(smiles)
        for _ in range(RANDOM_SMILES_PER_MOLECULE):
            rows.append(Chem.MolToSmiles(mol, canonical=False, doRandom=True))
    return rows


class SmilesDataset(Dataset):
    def __init__(self, smiles, tokenizer):
        self.smiles = list(smiles)
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.smiles)

    def __getitem__(self, index):
        encoded = self.tokenizer(
            self.smiles[index],
            max_length=MAX_LENGTH,
            truncation=True,
            padding="max_length",
            add_special_tokens=True,
            return_tensors="pt",
        )
        input_ids = encoded["input_ids"].squeeze(0)
        attention_mask = encoded["attention_mask"].squeeze(0)
        labels = input_ids.clone()
        labels[attention_mask == 0] = -100
        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "labels": labels,
        }


def validation_loss(model, loader):
    model.eval()
    losses = []
    with torch.no_grad():
        for batch in loader:
            batch = {name: value.to(device) for name, value in batch.items()}
            losses.append(model(**batch).loss.detach().cpu().item())
    model.train()
    return float(np.mean(losses))


def train_model(fraction, training_seed, train_rows, validation_rows, tokenizer):
    set_seed(training_seed)
    _, model = load_model()
    model.to(device)
    model.config.use_cache = False

    generator = torch.Generator().manual_seed(training_seed)
    train_loader = DataLoader(
        SmilesDataset(train_rows, tokenizer),
        batch_size=BATCH_SIZE,
        shuffle=True,
        generator=generator,
    )
    validation_loader = DataLoader(
        SmilesDataset(validation_rows, tokenizer),
        batch_size=16,
        shuffle=False,
    )
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)

    history = []
    best_loss = float("inf")
    best_step = 0
    checkpoint = WEIGHT_DIR / f"fraction_{fraction}_seed_{training_seed}.pt"
    step = 0

    while step < MAX_STEPS:
        for batch in train_loader:
            batch = {name: value.to(device) for name, value in batch.items()}
            optimizer.zero_grad(set_to_none=True)
            loss = model(**batch).loss
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            step += 1

            if step % EVAL_EVERY == 0:
                val_loss = validation_loss(model, validation_loader)
                history.append({
                    "fraction": fraction,
                    "training_seed": training_seed,
                    "step": step,
                    "train_loss": loss.detach().cpu().item(),
                    "validation_loss": val_loss,
                })
                if val_loss < best_loss:
                    best_loss = val_loss
                    best_step = step
                    torch.save(model.state_dict(), checkpoint)
                print(
                    "fraction:", fraction,
                    "| seed:", training_seed,
                    "| step:", step,
                    "| validation:", round(val_loss, 3),
                )
            if step == MAX_STEPS:
                break

    model.load_state_dict(torch.load(checkpoint, map_location=device, weights_only=True))
    model.config.use_cache = True
    model.eval()
    return model, pd.DataFrame(history), best_step, best_loss, checkpoint


def generate(model, tokenizer, generation_seed):
    set_seed(generation_seed)
    samples = []
    reached_limit = []
    prompt = torch.tensor([[tokenizer.bos_token_id]], device=device)

    with torch.no_grad():
        for start in tqdm(range(0, N_SAMPLES, 128), leave=False):
            batch_size = min(128, N_SAMPLES - start)
            output = model.generate(
                prompt.repeat(batch_size, 1),
                do_sample=True,
                temperature=TEMPERATURE,
                top_p=TOP_P,
                max_new_tokens=MAX_LENGTH - 1,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id,
                remove_invalid_values=True,
            )
            samples.extend(
                "".join(text.split())
                for text in tokenizer.batch_decode(output, skip_special_tokens=True)
            )
            reached_limit.extend(
                ~(output == tokenizer.eos_token_id).any(dim=1).cpu().numpy()
            )
    return samples, reached_limit

## Experiment


In [ ]:
tokenizer, pretrained_model = load_model()

supported_train = [s for s in train_order if roundtrip_ok(s, tokenizer)]
supported_validation = [s for s in validation_smiles if roundtrip_ok(s, tokenizer)]
pd.DataFrame({
    "smiles": train_order,
    "supported": [s in set(supported_train) for s in train_order],
}).to_csv(RESULT_DIR / "representation_coverage.csv", index=False)

print("Supported training molecules:", len(supported_train), "of", len(train_order))
print("Supported validation molecules:", len(supported_validation), "of", len(validation_smiles))

# Zero-shot sampling is independent of the FLP training fraction.
pretrained_model.to(device).eval()
for generation_seed in GENERATION_SEEDS:
    samples, reached = generate(pretrained_model, tokenizer, generation_seed)
    pd.DataFrame({
        "smiles": samples,
        "reached_max_length": reached,
    }).to_csv(RESULT_DIR / f"pretrained_generation_{generation_seed}.csv", index=False)
del pretrained_model
if torch.cuda.is_available():
    torch.cuda.empty_cache()

history_tables = []
checkpoint_rows = []
checkpoint_paths = []

for fraction, size in FRACTIONS.items():
    molecules = [s for s in train_order[:size] if s in set(supported_train)]
    train_rows = randomized_smiles(molecules, 42)
    train_rows = [s for s in train_rows if roundtrip_ok(s, tokenizer)]

    print()
    print("Fraction:", fraction, "| molecules:", len(molecules), "| SMILES:", len(train_rows))

    for training_seed in TRAINING_SEEDS:
        model, history, best_step, best_loss, checkpoint = train_model(
            fraction,
            training_seed,
            train_rows,
            supported_validation,
            tokenizer,
        )
        history_tables.append(history)
        checkpoint_paths.append(checkpoint)
        checkpoint_rows.append({
            "fraction": fraction,
            "training_seed": training_seed,
            "molecules": len(molecules),
            "training_smiles": len(train_rows),
            "best_step": best_step,
            "best_validation_loss": best_loss,
        })

        for generation_seed in GENERATION_SEEDS:
            samples, reached = generate(model, tokenizer, generation_seed)
            pd.DataFrame({
                "smiles": samples,
                "reached_max_length": reached,
                "fraction": fraction,
                "training_seed": training_seed,
                "generation_seed": generation_seed,
            }).to_csv(
                RESULT_DIR / f"fraction_{fraction}_train_{training_seed}_generation_{generation_seed}.csv",
                index=False,
            )

        del model
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

pd.concat(history_tables, ignore_index=True).to_csv(RESULT_DIR / "training_history.csv", index=False)
pd.DataFrame(checkpoint_rows).to_csv(RESULT_DIR / "checkpoints.csv", index=False)
print(pd.DataFrame(checkpoint_rows).round(4).to_string(index=False))

## Saved archives


In [ ]:
results_archive = shutil.make_archive("external_molgpt_results", "zip", RESULT_DIR)
weights_archive = Path("external_molgpt_weights.zip")
with ZipFile(weights_archive, "w") as archive:
    for path in checkpoint_paths:
        archive.write(path, arcname=path.name)

print("Results:", Path(results_archive).resolve())
print("Weights:", weights_archive.resolve())

try:
    from google.colab import files
    files.download(results_archive)
    files.download(weights_archive)
except ImportError:
    print("Kaggle stores the archives under Output.")